# Build a unified full-trajectory math/proof solver dataset

This notebook downloads and normalizes several math/proof datasets into one JSONL file for **full trajectory / full solution generation**.

## Target output schema

Each output example has this shape:

```json
{
  "problem": "...",
  "solution_steps": ["step 1", "step 2", "step 3"],
  "final_answer": "...",
  "example_index": 137,
  "source": "proofnet"
}
```

## Datasets handled here

- **DEMI-MathAnalysis**: expected as a local JSONL file you already exported.
- **ProofNet**: Hugging Face dataset `hoskinson-center/proofnet`.
- **NaturalProofs generation data**: Hugging Face dataset `wellecks/naturalproofs-gen`.
- **MATH**: Hugging Face dataset `EleutherAI/hendrycks_math`.

ProofNet contains **371** examples with a natural-language theorem statement and a natural-language proof. NaturalProofs is a larger corpus of theorem statements and proofs in natural mathematical language. MATH contains **12,500** competition problems with full step-by-step solutions. ─cite│turn743750search0│turn743750search9│turn303389search14━

## Notes

- Some source datasets are **proof-style** rather than answer-style. In those cases, `final_answer` may be empty.
- Solution steps are extracted heuristically from the original proof/solution text.
- The notebook is designed to be easy to edit if you want a stricter step-splitting strategy later.

In [ ]:
# Install older datasets to support dataset python scripts (removed in v3.0+)
!pip -q install "datasets<3.0" pandas tqdm

In [ ]:

from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

from datasets import load_dataset
from tqdm.auto import tqdm
import pandas as pd


In [ ]:

# -----------------------------
# Configuration
# -----------------------------

OUTPUT_DIR = Path("normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / "solver_full_trajectory_dataset.jsonl"

# GitHub DEMI URL (replace with the actual raw URL to the jsonl file)
DEMI_GITHUB_URL = "https://raw.githubusercontent.com/your-username/your-repo/main/demi_mathanalysis_sft.jsonl"

# Toggle sources
USE_DEMI = True
USE_PROOFNET = True
USE_NATURALPROOFS = True
USE_MATH = True

# Target Total Problems
TARGET_TOTAL_PROBLEMS = 1400

# MATH split choices:
MATH_SPLITS = ["train", "test"]

print(f"Output file will be written to: {OUTPUT_JSONL.resolve()}")


Output file will be written to: /content/normalized_outputs/solver_full_trajectory_dataset.jsonl


In [ ]:
from typing import Any, Dict, List, Optional
import re

# -----------------------------
# Utility helpers
# -----------------------------

def first_present(d: Dict[str, Any], keys: List[str], default=None):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return default

def clean_text(x: Any) -> str:
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()

def split_solution_into_steps(text: str) -> List[str]:
    """
    Heuristic step splitter:
    1. preserve numbered / bulleted lines
    2. otherwise split on double newlines
    3. fallback to sentence-ish chunks if needed
    """
    text = clean_text(text)
    if not text:
        return []

    # Normalize LaTeX line breaks a bit
    text = text.replace("\\\\\n", "\n").replace("\\\\", "\n")

    # Try line-based numbered/bulleted steps
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    numbered_like = []
    for ln in lines:
        if re.match(r"^(\(?\d+[\).\:]|[-*\u2022]|Step\s+\d+[:.\-])", ln, flags=re.I):
            numbered_like.append(re.sub(r"\s+", " ", ln).strip())

    if len(numbered_like) >= 2:
        return numbered_like

    # Split on paragraph breaks
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(paras) >= 2:
        return paras

    # Sentence-ish fallback
    sents = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\\(])", text)
    sents = [s.strip() for s in sents if s.strip()]
    if len(sents) >= 2:
        return sents

    return [text]

def extract_boxed_answer(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""

    # \boxed{...}
    m = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if m:
        return m[-1].strip()

    # final answer is ...
    patterns = [
        r"(?i)final answer\s*(?:is|:)\s*(.+)$",
        r"(?i)therefore[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)thus[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)answer\s*[:]\s*(.+)$",
    ]
    for pat in patterns:
        mm = re.search(pat, text, flags=re.M)
        if mm:
            return mm.group(1).strip().rstrip(".")

    return ""

def maybe_add(item: Optional[Dict[str, Any]], acc: List[Dict[str, Any]]):
    if item is None:
        return
    if not item.get("problem"):
        return
    if not item.get("solution_steps"):
        return
    acc.append(item)

def preview_dataset_dict(ds, name: str, n: int = 3):
    print(f"\n===== {name} =====")
    print(ds)
    if hasattr(ds, "keys"):
        for split in ds.keys():
            row = ds[split][0]
            print(f"Split={split}, columns={list(row.keys())}")
            break

## Source-specific normalizers

In [ ]:
def normalize_demi_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    """
    Tries to normalize a local DEMI JSONL export.
    Adjust the field mappings below if your local file uses different names.
    """
    problem = first_present(rec, ["problem", "question", "prompt", "input"])
    solution = first_present(
        rec,
        [
            "ground_truth_solution",
            "solution",
            "proof",
            "response",
            "output",
            "target",
            "completion",
            "answer",
        ],
    )
    final_answer = first_present(
        rec,
        ["final_answer", "ground_truth_answer", "answer_only", "short_answer"],
        default="",
    )

    problem = clean_text(problem)
    solution = clean_text(solution)
    final_answer = clean_text(final_answer) or extract_boxed_answer(solution)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(solution),
        "final_answer": final_answer,
        "example_index": example_index,
        "source": "demi_mathanalysis",
    }

def normalize_proofnet_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    problem = first_present(
        rec,
        [
            "nl_statement",
            "informal_statement",
            "natural_language_statement",
            "statement",
            "theorem",
        ],
    )
    proof = first_present(
        rec,
        [
            "nl_proof",
            "informal_proof",
            "natural_language_proof",
            "proof",
        ],
    )
    problem = clean_text(problem)
    proof = clean_text(proof)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(proof),
        "final_answer": "",
        "example_index": example_index,
        "source": "proofnet",
    }

def normalize_naturalproofs_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    """
    naturalproofs-gen has changed across versions / views, so this function is intentionally flexible.
    """
    problem = first_present(
        rec,
        [
            "statement",
            "theorem",
            "question",
            "prompt",
            "target_statement",
            "nl_statement",
            "text",
        ],
    )
    proof = first_present(
        rec,
        [
            "proof",
            "target",
            "response",
            "completion",
            "generated_proof",
            "nl_proof",
        ],
    )

    # Some variants package data in nested fields or instruction formats.
    if not problem:
        src = first_present(rec, ["source", "input", "context"], default="")
        if isinstance(src, dict):
            problem = first_present(src, ["statement", "theorem", "question", "prompt"])
    if not proof:
        tgt = first_present(rec, ["output", "answer"], default="")
        if isinstance(tgt, dict):
            proof = first_present(tgt, ["proof", "response", "target"])

    problem = clean_text(problem)
    proof = clean_text(proof)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(proof),
        "final_answer": "",
        "example_index": example_index,
        "source": "naturalproofs",
    }

def normalize_math_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    problem = clean_text(first_present(rec, ["problem", "question"]))
    solution = clean_text(first_present(rec, ["solution", "answer"]))
    final_answer = clean_text(first_present(rec, ["final_answer", "answer_only"], default=""))
    final_answer = final_answer or extract_boxed_answer(solution)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(solution),
        "final_answer": final_answer,
        "example_index": example_index,
        "source": "math",
        "level": first_present(rec, ["level"], default=""),
        "category": first_present(rec, ["type", "subject", "category"], default=""),
    }


## Load each dataset

In [ ]:

all_records = []
source_counts = {}
global_index = 0


In [ ]:

# -----------------------------
# DEMI (from GitHub Repository)
# -----------------------------
import subprocess
import json
import ast
from pathlib import Path

if USE_DEMI:
    demi_records = []
    print("Cloning DEMI-MathAnalysis repository...")
    # Clone the repo into /tmp
    subprocess.run(["git", "clone", "https://github.com/ziye2chen/DEMI-MathAnalysis.git", "/tmp/DEMI-MathAnalysis"], capture_output=True)

    repo_path = Path("/tmp/DEMI-MathAnalysis")
    # Search for any json or jsonl files
    data_files = list(repo_path.rglob("*.json")) + list(repo_path.rglob("*.jsonl"))
    print(f"Found data files: {[f.name for f in data_files]}")

    for data_file in data_files:
        try:
            with open(data_file, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if content.startswith("["):
                    # Parse as a single JSON array
                    try:
                        records = json.loads(content)
                    except json.JSONDecodeError:
                        records = ast.literal_eval(content)
                    demi_records.extend(records)
                else:
                    # Try parsing the entire content as a single dictionary first
                    try:
                        try:
                            record = json.loads(content)
                        except json.JSONDecodeError:
                            record = ast.literal_eval(content)
                        if isinstance(record, dict):
                            demi_records.append(record)
                        else:
                            raise ValueError("Not a single dictionary")
                    except Exception:
                        # Parse as JSONL
                        for line in content.split('\n'):
                            if line.strip():
                                try:
                                    demi_records.append(json.loads(line))
                                except json.JSONDecodeError:
                                    demi_records.append(ast.literal_eval(line))
        except Exception as e:
            print(f"Could not parse {data_file.name}: {e}")

    for rec in tqdm(demi_records, desc="Normalizing DEMI"):
        item = normalize_demi_record(rec, global_index)
        maybe_add(item, all_records)
        if item and item.get("problem") and item.get("solution_steps"):
            global_index += 1

    source_counts["demi_mathanalysis"] = len([r for r in all_records if r["source"] == "demi_mathanalysis"])
    print("Loaded DEMI examples:", source_counts["demi_mathanalysis"])
else:
    source_counts["demi_mathanalysis"] = 0
    print("Skipping DEMI.")


Cloning DEMI-MathAnalysis repository...
Found data files: ['problem_35.json', 'problem_86.json', 'problem_43.json', 'problem_23.json', 'problem_8.json', 'problem_81.json', 'problem_46.json', 'problem_52.json', 'problem_88.json', 'problem_53.json', 'problem_70.json', 'problem_83.json', 'problem_13.json', 'problem_69.json', 'problem_44.json', 'problem_57.json', 'problem_33.json', 'problem_80.json', 'problem_41.json', 'problem_5.json', 'problem_49.json', 'problem_25.json', 'problem_28.json', 'problem_31.json', 'problem_67.json', 'problem_51.json', 'problem_58.json', 'problem_34.json', 'problem_20.json', 'problem_4.json', 'problem_16.json', 'problem_42.json', 'problem_36.json', 'problem_14.json', 'problem_38.json', 'problem_66.json', 'problem_59.json', 'problem_85.json', 'problem_22.json', 'problem_75.json', 'problem_29.json', 'problem_87.json', 'problem_68.json', 'problem_73.json', 'problem_9.json', 'problem_47.json', 'problem_39.json', 'problem_72.json', 'problem_6.json', 'problem_76.jso

Normalizing DEMI:   0%|          | 0/176 [00:00<?, ?it/s]

Loaded DEMI examples: 176


In [ ]:

# -----------------------------
# ProofNet
# -----------------------------
if USE_PROOFNET:
    proofnet = load_dataset("hoskinson-center/proofnet", trust_remote_code=True)
    preview_dataset_dict(proofnet, "ProofNet")

    for split in proofnet.keys():
        rows = proofnet[split]
        # Using all examples, no debug limit
        for rec in tqdm(rows, desc=f"Normalizing ProofNet/{split}"):
            item = normalize_proofnet_record(rec, global_index)
            maybe_add(item, all_records)
            global_index += 1

    source_counts["proofnet"] = len([r for r in all_records if r["source"] == "proofnet"])
    print("Loaded ProofNet examples:", source_counts["proofnet"])
else:
    source_counts["proofnet"] = 0



===== ProofNet =====
DatasetDict({
    test: Dataset({
        features: ['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header'],
        num_rows: 186
    })
    validation: Dataset({
        features: ['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header'],
        num_rows: 185
    })
})
Split=test, columns=['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header']


Normalizing ProofNet/test:   0%|          | 0/186 [00:00<?, ?it/s]

Normalizing ProofNet/validation:   0%|          | 0/185 [00:00<?, ?it/s]

Loaded ProofNet examples: 369


In [ ]:

# -----------------------------
# NaturalProofs
# -----------------------------
# Target exactly 1000 proof-only problems from DEMI + ProofNet + NaturalProofs
TARGET_PROOF_PROBLEMS = 1000

# Calculate unique problems currently in all_records
current_unique = len(set((r.get("source", ""), r.get("problem", "")) for r in all_records))
naturalproofs_limit = max(0, TARGET_PROOF_PROBLEMS - current_unique)

if USE_NATURALPROOFS and naturalproofs_limit > 0:
    # Added verification_mode="no_checks" to bypass metadata mismatch errors without warnings
    naturalproofs = load_dataset("wellecks/naturalproofs-gen", trust_remote_code=True, verification_mode="no_checks")
    preview_dataset_dict(naturalproofs, "NaturalProofs")

    added_np = 0
    seen_np_problems = set()
    for split in naturalproofs.keys():
        rows = naturalproofs[split]
        for rec in tqdm(rows, desc=f"Normalizing NaturalProofs/{split}"):
            if added_np >= naturalproofs_limit:
                break
            item = normalize_naturalproofs_record(rec, global_index)

            # Check if valid item before adding, and enforce uniqueness on the fly
            if item and item.get("problem") and item.get("solution_steps"):
                prob_key = item["problem"]
                if prob_key not in seen_np_problems:
                    seen_np_problems.add(prob_key)
                    all_records.append(item)
                    global_index += 1
                    added_np += 1

        if added_np >= naturalproofs_limit:
            break

    source_counts["naturalproofs"] = len([r for r in all_records if r["source"] == "naturalproofs"])
    print("Loaded NaturalProofs examples:", source_counts["naturalproofs"])
else:
    source_counts["naturalproofs"] = 0
    print(f"Skipping NaturalProofs. (limit: {naturalproofs_limit})")



===== NaturalProofs =====
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 12424
    })
    validation: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 1139
    })
    test: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 1135
    })
})
Split=train, columns=['id', 'title', 'text', 'target', 'ctxs']


Normalizing NaturalProofs/train:   0%|          | 0/12424 [00:00<?, ?it/s]

Loaded NaturalProofs examples: 543


In [ ]:

# -----------------------------
# MATH
# -----------------------------
# The remaining problems to hit 1200 total come from MATH
remaining_for_math = 200

if USE_MATH and remaining_for_math > 0:
    MATH_CONFIGS = [
        'algebra', 'counting_and_probability', 'geometry',
        'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
    ]

    added_math = 0
    for config in MATH_CONFIGS:
        if added_math >= remaining_for_math:
            break

        math_ds = load_dataset("EleutherAI/hendrycks_math", config, trust_remote_code=True)

        for split in MATH_SPLITS:
            if split not in math_ds:
                continue

            rows = math_ds[split]
            for rec in tqdm(rows, desc=f"Normalizing MATH/{config}/{split}"):
                if added_math >= remaining_for_math:
                    break
                item = normalize_math_record(rec, global_index)
                if item and item.get("problem") and item.get("solution_steps"):
                    all_records.append(item)
                    global_index += 1
                    added_math += 1

    source_counts["math"] = len([r for r in all_records if r["source"] == "math"])
    print("Loaded MATH examples:", source_counts["math"])
else:
    source_counts["math"] = 0
    print(f"Skipping MATH. (limit: {remaining_for_math})")


Normalizing MATH/algebra/train:   0%|          | 0/1744 [00:00<?, ?it/s]

Normalizing MATH/algebra/test:   0%|          | 0/1187 [00:00<?, ?it/s]

Loaded MATH examples: 200


## Basic cleanup and export

In [ ]:

# Deduplicate loosely by (problem, source)
seen = set()
deduped = []
for rec in all_records:
    key = (rec.get("source", ""), rec.get("problem", ""))
    if key in seen:
        continue
    seen.add(key)
    deduped.append(rec)

all_records = deduped

for i, rec in enumerate(all_records):
    rec["example_index"] = i

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(all_records):,} examples to {OUTPUT_JSONL}")
print("Counts by source:")
print(pd.Series([r["source"] for r in all_records]).value_counts())


Wrote 1,200 examples to normalized_outputs/solver_full_trajectory_dataset.jsonl
Counts by source:
naturalproofs        543
proofnet             369
math                 200
demi_mathanalysis     88
Name: count, dtype: int64


In [ ]:
proof_only = [r for r in all_records if r["source"] in {"demi_mathanalysis", "proofnet", "naturalproofs"}]
with open(OUTPUT_DIR / "solver_full_trajectory_proof_only.jsonl", "w", encoding="utf-8") as f:
    for rec in proof_only:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(len(proof_only))

1000


## Inspect a few normalized examples

In [ ]:

for rec in all_records[:5]:
    print("=" * 100)
    print("source:", rec["source"])
    print("problem:", rec["problem"][:400])
    print("num_steps:", len(rec["solution_steps"]))
    print("first_step:", rec["solution_steps"][0][:300] if rec["solution_steps"] else "")
    print("final_answer:", rec.get("final_answer", ""))


source: demi_mathanalysis
problem: Show that the maximum of $\left| Q^{(n)}(x) \right|$ over $[-1, 1]$ is equal to $2^n n!$, where
\[
Q(x) = (1 - x^2)^n.
\]
num_steps: 3
first_step: By Cauchy's integral formula we get
\[
Q^{(n)}(x) = \frac{n!}{2\pi i} \int_C \frac{(1 - z^2)^n}{(z - x)^{n+1}} \, dz
\]
where \(C\) is an oriented circle centered at \(z = x\) with radius \(r > 0\). Hence putting
\[
z = x + re^{i\theta}
\]
for \(0 \leq \theta < 2\pi\) we obtain
\[
Q^{(n)}(x) = \frac
final_answer: 
source: demi_mathanalysis
problem: Let $s > -1$ be a real number. Suppose that $f \in C[0, \infty)$ is a convex function having the piecewise continuous derivative $f'(x)$ and satisfying $f(0) \geq 0$. Suppose further that $f'(0+)$ exists when $f(0) = 0$. Then show that
\[
\int_0^\infty x^s \exp\left(-\frac{f(x)}{x}\right) dx \leq \int_0^\infty x^s \exp\left(-f'\left(\frac{x}{e}\right)\right) dx.
\]
Prove moreover that the constant
num_steps: 4
first_step: As is shown in \textbf{Solution 9.6}, the


## Optional: stricter filtering for proof-style training only

If you want a more proof-heavy export, you can keep only `proofnet`, `naturalproofs`, and `demi_mathanalysis`:

```python
proof_only = [r for r in all_records if r["source"] in {"demi_mathanalysis", "proofnet", "naturalproofs"}]
with open(OUTPUT_DIR / "solver_full_trajectory_proof_only.jsonl", "w", encoding="utf-8") as f:
    for rec in proof_only:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(len(proof_only))
```

## Optional: add train/val split

```python
import random
random.seed(42)
records = all_records[:]
random.shuffle(records)
cut = int(0.98 * len(records))
train_records = records[:cut]
val_records = records[cut:]
```


In [ ]:
import json
from pathlib import Path

# Deduplicate loosely by (problem, source) to handle any rerun accumulation
seen = set()
deduped = []
for rec in all_records:
    key = (rec.get("source", ""), rec.get("problem", ""))
    if key in seen:
        continue
    seen.add(key)
    deduped.append(rec)

all_records = deduped

sft_records = []
for rec in all_records:
    # Build the full trajectory response
    steps = rec.get("solution_steps", [])
    trajectory = "\n".join(steps)
    if rec.get("final_answer"):
        trajectory += f"\n\nFinal Answer: {rec['final_answer']}"

    sft_example = {
        "messages": [
            {"role": "user", "content": rec["problem"]},
            {"role": "assistant", "content": trajectory}
        ],
        "source": rec["source"]
    }
    sft_records.append(sft_example)

SFT_OUTPUT_JSONL = OUTPUT_DIR / "solver_full_trajectory_sft.jsonl"
with open(SFT_OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in sft_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(sft_records):,} SFT examples to {SFT_OUTPUT_JSONL}")

import pandas as pd
print("\nSFT counts by source:")
print(pd.Series([r["source"] for r in sft_records]).value_counts())

Wrote 1,200 SFT examples to normalized_outputs/solver_full_trajectory_sft.jsonl

SFT counts by source:
naturalproofs        543
proofnet             369
math                 200
demi_mathanalysis     88
Name: count, dtype: int64
